# Data Profiling

El perfilado de datos o comunmente referenciado "data profiling" es un conjunto de técnicas en cuales se evaluan conjuntos de datos para lograr clasificar su utilidad en el problema abordado. Factores que se evaluan comunmente son la precisión,coherencia y relevancia al problema.

Esta metodología tiene relevancia en proyectos que impliquen **business intelligence**, **almacenamiento de datos** o **big data** ya que su objetivo es lograr identificar la calidad de datos con la que estamos trabajando. Generalmente existen 3 perspectivas en cuanto el perfilado de datos: 

- **Estructuras:** La idea central de este enfoque es garantizar la coherencia de los datos verificando el formato y esquema que deben seguir. 

- **Contenido:** Se busca principalmente errores o problemas sistémicos ya sean valores incorrectos o nulos.

- **Relaciones:** Esta perspectiva se enfoca en encontrar la relación entre cada conjunto de datos relevante al proyecto. Esto se logra realizando un análisis de metadatos.

Nosotros nos estaremos enfocando en las perspectivas de estructura y contenido y definiremos qué criterios se utilizaran para perfiliar los datos con los que estaremos trabajando. Para manejar la calidad y perfilado de datos, nos apoyaremos del framework de [`great_expectations`](https://docs.greatexpectations.io/docs/core/introduction/). Entonces, comenzamos estableciendo un contexto 

## Great Expectations

In [1]:
import great_expectations as gx 

context = gx.get_context()
print(type(context).__name__)

EphemeralDataContext


Ahora, necesitamos definir una fuente de datos. En nuestro caso, estamos manejando archivos locales

In [2]:
data_source_name = "local_raw" 
data_source = context.data_sources.add_pandas(name=data_source_name)


In [3]:
data_asset_name = "ventas_producto"
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

In [4]:
batch_definition_name = "ventas_diarias"
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

In [5]:
import pandas as pd

dataset_name = "facturas_ventas.parquet"
data_path = f"../data/raw/{dataset_name}"
dataframe = pd.read_parquet(data_path)
batch_parameters = {"dataframe": dataframe}

In [6]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="month", max_value=12, min_value=1
)

batch = batch_definition.get_batch(batch_parameters=batch_parameters)


validation_results = batch.validate(expectation)
print(validation_results)

Calculating Metrics:  30%|███       | 3/10 [00:00<00:00, 870.31it/s] 

{
  "success": false,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "column": "month",
      "min_value": 1.0,
      "max_value": 12.0,
      "batch_id": "local_raw-ventas_producto"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {},
  "meta": {},
  "exception_info": {
    "MetricConfigurationID(metric_name='column_values.between.condition', metric_domain_kwargs_id='700d81808aa33d4745aa60e97f6ff87d', metric_value_kwargs_id='b2393531b226632fcdbf25e4e8e2e3cf')": {
      "exception_traceback": "Traceback (most recent call last):\n  File \"c:\\Users\\federico.cirett\\Documents\\ct_dashboard\\mlops\\.venv\\Lib\\site-packages\\great_expectations\\execution_engine\\execution_engine.py\", line 577, in _process_direct_and_bundled_metric_computation_configurations\n    metric_computation_configuration.metric_fn(  # type: ignore[misc] # F not callable\n  File \"c:\\Users\\federico.cirett\\Documents\\ct_dashboard\\mlops\\.venv\

In [7]:
dataframe.head()

,productId,quantity,date,price,clientId,folio,description,cost,branchId,storageId,branch,homoclave,state,clave,category
0,MEMDAT6260,1,2025-01-01,650.304328,GDL2988,DFP218209,DDR5 UDIMM 16GB 4800MHZ AD5U480016G-S,2553.0225,DFP,34A,"MEXICO, PALACIO DE LOS DEPORTES",DFP,DISTRITO FEDERAL,MEMDAT6260,Memorias RAM
1,MEMDAT6200,2,2025-01-01,407.702063,GDL2988,HMO1076454,DDR4 8GB 3600MHZ AX4U36008G18I-ST50,744.23211,HMO,01A,"HERMOSILLO, SON.",HMO,SONORA,MEMDAT6200,Memorias RAM
2,BOCRBT510,1,2025-01-02,55.623071,LE2416,LEO425090,AUDIFONOS C/MICROFONO VERDE 651428 RBT,52.342034,LEO,11A,"LEON, GUANAJUATO",LEO,GUANAJUATO,BOCRBT510,Auriculares
3,MOUTCH940,1,2025-01-02,41.65625,QRO2284,QRO440686,MOUSE ALAMBRICO USB TZACMOA01 TECHZONE,37.5,QRO,09A,"QUERETARO, QUERETARO",QRO,QUERÉTARO,MOUTCH940,Mouse
4,RCKACC1430,1,2025-01-02,286.406491,LE2416,LEO425090,790055 PLUG RJ45 CAT 5E UTP MULTIFILAR 1,326.004322,LEO,11A,"LEON, GUANAJUATO",LEO,GUANAJUATO,RCKACC1430,Adaptadores para Red


In [8]:
suite_name = f"{data_source_name}.{dataset_name}.validation.dev"
suite = gx.ExpectationSuite(name=suite_name)
suite = context.suites.add(suite)

In [9]:
positive_units_sold_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="quantity", min_value=0,strict_min=True
)
positive_cost_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="cost", min_value=0,strict_min=True
)
positive_price_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="price", min_value=0,strict_min=True
)

suite.add_expectation(positive_units_sold_expectation)
suite.add_expectation(positive_cost_expectation)
suite.add_expectation(positive_price_expectation)

ExpectColumnValuesToBeBetween(id='5c884b89-d2f0-46db-a4b5-7d44ae7de397', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='price', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=None, strict_min=True, strict_max=False)

In [10]:
pipeline_stage = "ingestion"
validation_definition_name = f"{dataset_name}.{pipeline_stage}.{suite_name}.dev"
validation_definition = gx.ValidationDefinition(
    data=batch_definition, suite=suite, name=validation_definition_name
)
validation_definition = context.validation_definitions.add(validation_definition)

In [11]:
validation_results = validation_definition.run(batch_parameters=batch_parameters)
print(validation_results)


Calculating Metrics: 100%|██████████| 24/24 [00:00<00:00, 31.96it/s]

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_between",
        "kwargs": {
          "batch_id": "local_raw-ventas_producto",
          "column": "quantity",
          "min_value": 0.0,
          "strict_min": true
        },
        "meta": {},
        "id": "fe671eee-f306-4cea-b9e4-5097a5189ce7",
        "severity": "critical"
      },
      "result": {
        "element_count": 762246,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "missing_count": 0,
        "missing_percent": 0.0,
        "unexpected_percent_total": 0.0,
        "unexpected_percent_nonmissing": 0.0,
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    

## Perfilado de columnas

Se analizan las tablas y se cuenta el número de veces que aparece cada valor dentro de cada columna. 

In [15]:
import pandas as pd
import great_expectations as gx 


context = gx.get_context()
data_source_name = "local_processed" 
data_source = context.data_sources.add_pandas(name=data_source_name)

data_asset_name = "ventas_producto"
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

batch_definition_name = "ventas_diarias"
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

dataset_name = "HERMOSILLO, SON/MEMSTY050.parquet"
data_path = f"../data/processed/{dataset_name}"
dataframe = pd.read_parquet(data_path)
batch_parameters = {"dataframe": dataframe}
batch = batch_definition.get_batch(batch_parameters=batch_parameters)

suite_name = f"{data_source_name}.{dataset_name}.validation.dev"
suite = gx.ExpectationSuite(name=suite_name)
suite = context.suites.add(suite)

In [17]:
productId_exists_expectation = gx.expectations.ExpectColumnToExist(column="productId")
quantity_column_exists_expectation = gx.expectations.ExpectColumnToExist(column="quantity")
date_column_exists_expectation = gx.expectations.ExpectColumnToExist(column="date")
clientId_exists_expectation = gx.expectations.ExpectColumnToExist(column="clientId")

suite.add_expectation(productId_exists_expectation)
suite.add_expectation(quantity_column_exists_expectation)
suite.add_expectation(date_column_exists_expectation)
suite.add_expectation(clientId_exists_expectation)

ExpectColumnToExist(id='e48bc4e0-bac1-4f47-874b-2dd8281619b9', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=False, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='clientId', column_index=None)

## Perfilado entre columnas

El proceso de análisis de claves analiza la matriz de valores de atributos buscando una posible clave principal. Mientras que el proceso de análisis de dependencias funciona para identificar qué relaciones o patrones están integrados en el conjunto de datos.

In [18]:
def calculate_iqr_bounds(sales_series:pd.Series)->tuple[float,float]:
    """
    Calcula los valores de las fronteras intercuántilicas de valores introducidos.

    Parametros:
    - sales_series: pandas.Series, Serie de valores, típicamente la cantidad de unidades vendidas
    Regresa:
    - (valor inferior , valor superior) : tuple[float,float] , Valores de las fronteras intercuantílicas
    
    """
    q1 = sales_series.quantile(0.25)
    q3 = sales_series.quantile(0.75)
    iqr = q3 - q1
    return (q1 - 1.5 * iqr, q3 + 1.5 * iqr)


outlier_threshold = calculate_iqr_bounds(dataframe["quantity"])
outlier_units_sold_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="quantity", min_value=outlier_threshold[0],max_value=outlier_threshold[1], strict_min=True
)

suite.add_expectation(outlier_units_sold_expectation)


ExpectColumnValuesToBeBetween(id='74f6ddcc-8043-4157-abce-ed2345129cbf', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='quantity', mostly=1, row_condition=None, condition_parser=None, min_value=np.float64(-125.0), max_value=np.float64(235.0), strict_min=True, strict_max=False)

## Validación de reglas de datos
Evalúa los conjuntos de datos en comparación con las reglas y estándares establecidos para verificar que de hecho están siguiendo esas reglas predefinidas

In [19]:
positive_units_sold_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="quantity", min_value=0,strict_min=True
)
positive_units_sold_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="quantity", min_value=0,strict_min=True
)
positive_cost_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="cost", min_value=0,strict_min=True
)
positive_price_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="price", min_value=0,strict_min=True
)

suite.add_expectation(positive_units_sold_expectation)
suite.add_expectation(positive_cost_expectation)
suite.add_expectation(positive_price_expectation)

ExpectColumnValuesToBeBetween(id='4eb2983b-61fa-4aef-b303-becd24cf8617', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='price', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=None, strict_min=True, strict_max=False)

In [20]:
pipeline_stage = "staging"
validation_definition_name = f"{dataset_name}.{pipeline_stage}.{suite_name}.dev"
validation_definition = gx.ValidationDefinition(
    data=batch_definition, suite=suite, name=validation_definition_name
)
validation_definition = context.validation_definitions.add(validation_definition)

validation_results = validation_definition.run(batch_parameters=batch_parameters)
print(validation_results)

Calculating Metrics: 100%|██████████| 29/29 [00:00<00:00, 2715.79it/s]

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_to_exist",
        "kwargs": {
          "batch_id": "local_processed-ventas_producto",
          "column": "productId"
        },
        "meta": {},
        "id": "84606a5e-85e2-4247-8228-552969cd1871",
        "severity": "critical"
      },
      "result": {},
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_to_exist",
        "kwargs": {
          "batch_id": "local_processed-ventas_producto",
          "column": "quantity"
        },
        "meta": {},
        "id": "c6f55545-a7bb-4081-9992-c44d2d129dd3",
        "severity": "critical"
      },
      "result": {},
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "e

In [ ]:
from great_expectations.expectations import UnexpectedRowsExpectation

for evr in validation_results.results:
    # Filtrar por estado porque get_unexpected_rows() solo contempla 'UnexpectedRowsExpectation'
    if not evr.success and isinstance(evr.expectation, UnexpectedRowsExpectation):
        unexpected_rows = validation_definition.get_unexpected_rows(
            evr.expectation,
            batch_parameters=validation_results.batch_parameters,
        )
        print(f"{len(unexpected_rows)} unexpected rows found")

## Referencias 
[¿Qué es el perfilado de datos?](https://www.ibm.com/mx-es/think/topics/data-profiling)

[Importancia del perfilado de datos, tipos y herramientas](https://datos.gob.es/es/blog/importancia-del-perfilado-de-datos-tipos-y-herramientas)
[Great Expectations documentation](https://docs.greatexpectations.io/docs/home)